In [ ]:
import pandas as pd
import pybedtools
import numpy as np
from pathlib import Path

In [3]:
def format_peaks(peak_ids: pd.Series | pd.Index) -> pd.DataFrame:
    """
    Splits peaks from `chrN:start-end` or `chrN-start-end` format into a DataFrame.
    
    Creates a dataframe with the following columns:
    1) "peak_id": peakN+1 where N is the index position of the peak
    2) "chromosome": chrN
    3) "start"
    4) "end"
    5) "strand": List of "." values, we dont have strand information for our peaks.
    
    Args:
        peak_ids (pd.Series):
            Series containing the peak locations in "chrN:start-end" or "chrN-start-end" format.
            
    Returns:
        peak_df (pd.DataFrame):
            DataFrame of peak locations in the correct format for Homer and the sliding window method
    """
    if peak_ids.empty:
        raise ValueError("Input peak ID list is empty.")
    
    peak_ids = pd.Series(peak_ids).drop_duplicates().astype(str).str.strip()

    # Keep only canonical chromosomes/records that contain "chr".
    peak_ids = peak_ids[peak_ids.str.contains("chr", regex=False)]
    if peak_ids.empty:
        raise ValueError("No peak IDs containing 'chr' were found after filtering.")

    # Primary parse: supports strings containing a canonical peak token anywhere,
    # such as "chr1:100-200", "hg38.chr1:100-200", or "hg38.chr1-100-200".
    parsed = peak_ids.str.extract(
        r'.*?(?P<chromosome>chr[^\s:-]+)(?::|-)(?P<start>\d+)-(?P<end>\d+)\s*$'
    )

    # Fallback parse: BED-like row strings, e.g. "hg38.chr1 100 200 ...".
    missing = parsed["chromosome"].isnull() | parsed["start"].isnull() | parsed["end"].isnull()
    if missing.any():
        bed_like = peak_ids[missing].str.extract(
            r'^\s*(?P<chromosome>\S+)\s+(?P<start>\d+)\s+(?P<end>\d+)(?:\s|$)'
        )
        parsed.loc[missing, ["chromosome", "start", "end"]] = bed_like[["chromosome", "start", "end"]].values

    # Normalize chromosomes like "hg38.chr1" to "chr1" by removing any prefix before "chr".
    parsed["chromosome"] = parsed["chromosome"].str.extract(r'(chr[^\s:]*)$', expand=False)

    if parsed["chromosome"].isnull().any() or parsed["start"].isnull().any() or parsed["end"].isnull().any():
        bad_examples = peak_ids[
            parsed["chromosome"].isnull() | parsed["start"].isnull() | parsed["end"].isnull()
        ].head(3).tolist()
        raise ValueError(
            "Malformed peak IDs. Expect one of: 'chr:start-end', 'chr-start-end', "
            "or BED-like 'chrom start end ...'. "
            f"Examples that failed parsing: {bad_examples}"
        )

    peak_df = pd.DataFrame({
        # "peak_id": [f"peak{i + 1}" for i in range(len(peak_ids))],
        "chromosome": parsed["chromosome"],
        "start": pd.to_numeric(parsed["start"], errors='coerce').astype(int),
        "end": pd.to_numeric(parsed["end"], errors='coerce').astype(int),
        "strand": ["."] * len(peak_ids)
    })
    
    peak_df["peak_id"] = (
        peak_df["chromosome"].astype(str) + ":" +
        peak_df["start"].astype(str) + "-" +
        peak_df["end"].astype(str)
    )
    
    return peak_df

def find_genes_near_peaks(
    peak_bed: pybedtools.BedTool, 
    tss_bed: pybedtools.BedTool, 
    tss_distance_cutoff: int | float = 1e6
    ):
    """
    Identify genes whose transcription start sites (TSS) are near scATAC-seq peaks.
    
    This function:
        1. Uses BedTools to find peaks that are within tss_distance_cutoff bp of each gene's TSS.
        2. Converts the BedTool result to a pandas DataFrame.
        3. Computes the absolute distance between the peak end and gene start (as a proxy for TSS distance).
        
    Args:
        peak_bed (pybedtools.BedTool):
            BedTool object representing scATAC-seq peaks.
        tss_bed (pybedtools.BedTool):
            BedTool object representing gene TSS locations.
        tss_distance_cutoff (int): 
            The maximum distance (in bp) from a TSS to consider a peak as potentially regulatory.
        
    Returns:
        peak_tss_subset_df (pandas.DataFrame): 
            A DataFrame containing columns "peak_id", "target_id", and the scaled TSS distance "TSS_dist"
            for peak–gene pairs.
    """
    
    peak_tss_overlap = peak_bed.window(tss_bed, w=tss_distance_cutoff)
    
    cols = [
        "peak_chr", "peak_start", "peak_end", "peak_id",
        "gene_chr", "gene_start", "gene_end", "gene_id"
    ]
    peak_tss_overlap_df = peak_tss_overlap.to_dataframe(
        names=cols,
        low_memory=False
    )

    # Coerce numeric cols safely & drop malformed rows
    for c in ["peak_start", "peak_end", "gene_start", "gene_end"]:
        peak_tss_overlap_df[c] = pd.to_numeric(peak_tss_overlap_df[c], errors="coerce")
    peak_tss_overlap_df = peak_tss_overlap_df.dropna(subset=["peak_start", "peak_end", "gene_start", "gene_end"]).copy()
        
    # Calculate the absolute distance in basepairs between the peak's end and gene's start.
    distances = np.abs(peak_tss_overlap_df["peak_end"].values - peak_tss_overlap_df["gene_start"].values)
    peak_tss_overlap_df["TSS_dist"] = distances
    
    # Sort by the TSS distance (lower values imply closer proximity and therefore stronger association)
    peak_tss_overlap_df = peak_tss_overlap_df.sort_values("TSS_dist")
    
    return peak_tss_overlap_df

In [ ]:
DATA_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")

organism_code = "mm10"
dataset_name = "mouse_liver"
sample_name = "sample_1"

tss_bed_file = DATA_DIR / "genome_data" / "genome_annotation" / organism_code / "gene_tss.bed"
genome_file = genome_file = DATA_DIR / "genome_data" / "reference_genome" / organism_code / f"{organism_code}.chrom.sizes"

atac_file = DATA_DIR / "processed" / dataset_name / sample_name / "RE_pseudobulk.parquet"
atac_df = pd.read_parquet(atac_file)

max_peak_distance = 100_000

peak_locs_df = format_peaks(pd.Series(atac_df.index)).rename(columns={"chromosome": "chrom"})
peak_bed = pybedtools.BedTool.from_dataframe(
    peak_locs_df[["chrom", "start", "end", "peak_id"]]
)

tss_bed = pybedtools.BedTool(str(tss_bed_file))

# Step 3: Find peaks near TSS and compute distances
genes_near_peaks = find_genes_near_peaks(peak_bed, tss_bed, tss_distance_cutoff=max_peak_distance)

genes_near_peaks = genes_near_peaks.rename(columns={"gene_id": "target_id"})
genes_near_peaks["target_id"] = genes_near_peaks["target_id"].str.upper()

if "TSS_dist" not in genes_near_peaks.columns:
    raise ValueError("Expected column 'TSS_dist' missing from find_genes_near_peaks output.")

# Step 5: Drop rows where the peak is too far from the gene
genes_near_peaks: pd.DataFrame = genes_near_peaks[genes_near_peaks["TSS_dist"] <= max_peak_distance]

# Save the peak to TSS distance file for this sample
peak_to_tss_dist_file = DATA_DIR / "processed" / dataset_name / sample_name / "peak_to_gene_dist.parquet"
Path(peak_to_tss_dist_file).parent.mkdir(parents=True, exist_ok=True)
genes_near_peaks.to_parquet(peak_to_tss_dist_file, index=False)

In [10]:


def download_chip_atlas_cell_type(
    genome, 
    track_type_class="Oth",
    cell_type_class="ALL", 
    threshold="05", 
    track_type="AllAg", 
    cell_type_specific="AllCell",
    timeout=30,
    ground_truth_dir = Path(DATA_DIR) / "ground_truth_files",
    force_redownload=False,
    ):
    
    file_string = f"{track_type_class}.{cell_type_class}.{threshold}.{track_type}.{cell_type_specific}.bed"
    
    output_path = ground_truth_dir / file_string
    
    if not output_path.exists() or force_redownload:
    
        url = f"https://chip-atlas.dbcls.jp/data/{genome}/assembled/{file_string}"

        print(f"Downloading ChIP-Atlas data from: {file_string}")

        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()

                total_size = int(r.headers.get("content-length", 0))
                

                with open(output_path, "wb") as f:
                    with tqdm(
                        total=total_size,
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        desc="Downloading",
                    ) as pbar:
                        for chunk in r.iter_content(chunk_size=8192):
                            if chunk:  # skip keep-alive chunks
                                f.write(chunk)
                                pbar.update(len(chunk))

        except Exception as e:
            print(f"Error fetching ChIP-Atlas data: {e}")
    else:
        print(f"File already exists at {output_path}, skipping download.")

download_chip_atlas_cell_type(
    genome="mm10",
    track_type_class="Oth",
    cell_type_class="Liv",
    threshold="05",
    track_type="AllAg",
    cell_type_specific="Liver"
)


Downloading: 100%|█████████████████████████████████████████████████████████████████████| 4.47G/4.47G [04:41<00:00, 17.0MB/s]
